# NETRA FICN model training and export

This notebook trains a **research-only** Indian-currency counterfeit classifier and exports `active_model.keras` plus `active_model_card.json`, directly loadable by RAKSHA NETRA. It does **not** make an RBI certification claim. Use only data whose licence and provenance permit your intended use.

Default candidate: Kaggle `preetrank/indian-currency-real-vs-fake-notes-dataset` (CC BY-NC-SA 4.0). For police, bank, or commercial deployment, replace it with an RBI/agency-authorised FICN collection and update the provenance fields below.

In [ ]:
# Configuration
DATASET_MODE = 'kaggle_research'  # or 'authorised_upload'
KAGGLE_DATASET = 'preetrank/indian-currency-real-vs-fake-notes-dataset'
DATASET_NAME = 'indian-real-fake-notes-kaggle-v1'
DATASET_URL = 'https://www.kaggle.com/datasets/preetrank/indian-currency-real-vs-fake-notes-dataset'
DATASET_LICENSE = 'CC BY-NC-SA 4.0'
# Enter your authorised collection reference when DATASET_MODE='authorised_upload'.
AUTHORISED_SOURCE_CASE = 'replace-with-approved-data-sharing-reference'
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
MAX_FALSE_POSITIVE_RATE = 0.05
MIN_VALIDATION_ACCURACY = 0.90


In [ ]:
# Download the research candidate, or upload an authorised ZIP structured as real/<denomination>/... and fake/<denomination>/...
from pathlib import Path
import shutil, os
ROOT = Path('/content/netra_training')
RAW = ROOT / 'raw'
RAW.mkdir(parents=True, exist_ok=True)
if DATASET_MODE == 'kaggle_research':
    !pip -q install kaggle
    from google.colab import files
    if not Path('/root/.kaggle/kaggle.json').exists():
        print('Upload your Kaggle API token (kaggle.json). It is used only in this Colab runtime.')
        uploaded = files.upload()
        Path('/root/.kaggle').mkdir(exist_ok=True)
        shutil.move(next(iter(uploaded)), '/root/.kaggle/kaggle.json')
        os.chmod('/root/.kaggle/kaggle.json', 0o600)
    !kaggle datasets download -d $KAGGLE_DATASET -p $RAW --unzip
else:
    from google.colab import files
    print('Upload the authorised dataset ZIP. Do not upload seized-note images unless your approval permits it.')
    uploaded = files.upload()
    archive = ROOT / next(iter(uploaded))
    shutil.move(next(iter(uploaded.keys())), archive)
    shutil.unpack_archive(archive, RAW)


In [ ]:
# Build a manifest; labels are inferred only from explicit real/authentic and fake/counterfeit folders.
import hashlib, json, re
from collections import Counter
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
def label_and_denomination(path):
    parts = [p.lower() for p in path.relative_to(RAW).parts]
    label = next(('counterfeit' for p in parts if p in {'fake', 'counterfeit', 'ficn'}), None)
    label = label or next(('authentic' for p in parts if p in {'real', 'genuine', 'authentic'}), None)
    denomination = next((re.sub(r'[^0-9]', '', p) for p in parts if re.fullmatch(r'[₹rs._ -]*[0-9]{2,4}[₹rs._ -]*', p)), 'unknown')
    return label, denomination
records, seen = [], set()
for file in RAW.rglob('*'):
    if not file.is_file() or file.suffix.lower() not in IMAGE_EXTENSIONS:
        continue
    label, denomination = label_and_denomination(file)
    if label is None:
        continue
    digest = hashlib.sha256(file.read_bytes()).hexdigest()
    if digest in seen:  # avoids exact duplicate leakage into hold-out metrics
        continue
    seen.add(digest)
    records.append({'path': str(file), 'label': label, 'denomination': denomination, 'contentSha256': digest})
print('usable records:', len(records), Counter(r['label'] for r in records), Counter(r['denomination'] for r in records))
assert len(records) >= 100 and {r['label'] for r in records} == {'authentic', 'counterfeit'}, 'Need at least 100 de-duplicated, labelled images and both classes.'
MANIFEST = ROOT / 'manifest.jsonl'
MANIFEST.write_text('\n'.join(json.dumps(r, sort_keys=True) for r in records), encoding='utf-8')
MANIFEST_SHA256 = hashlib.sha256(MANIFEST.read_bytes()).hexdigest()
print('manifest SHA-256:', MANIFEST_SHA256)


In [ ]:
# Stratified 70/15/15 split. Exact-content duplicates were removed above.
!pip -q install scikit-learn
import numpy as np, tensorflow as tf
from sklearn.model_selection import train_test_split
paths = np.array([r['path'] for r in records])
labels = np.array([1 if r['label'] == 'counterfeit' else 0 for r in records], dtype=np.float32)
train_p, temp_p, train_y, temp_y = train_test_split(paths, labels, test_size=0.30, stratify=labels, random_state=2026)
val_p, test_p, val_y, test_y = train_test_split(temp_p, temp_y, test_size=0.50, stratify=temp_y, random_state=2026)
def decode(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image = tf.image.resize(tf.cast(image, tf.float32), IMAGE_SIZE) / 255.0
    return image, label
def dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels)).map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    if training: ds = ds.shuffle(len(paths), seed=2026)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
train_ds, val_ds, test_ds = dataset(train_p, train_y, True), dataset(val_p, val_y), dataset(test_p, test_y)
print({'train': len(train_p), 'validation': len(val_p), 'holdout': len(test_p)})


In [ ]:
# Compact CNN. The deployed NETRA runtime supplies RGB pixels scaled to [0, 1], matching this model.
tf.keras.utils.set_random_seed(2026)
augment = tf.keras.Sequential([tf.keras.layers.RandomRotation(0.03), tf.keras.layers.RandomContrast(0.08), tf.keras.layers.RandomZoom(0.05)])
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(*IMAGE_SIZE, 3)), augment,
    tf.keras.layers.Conv2D(32, 3, activation='relu'), tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'), tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu'), tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.35), tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])
callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)


In [ ]:
# Select a threshold on validation data, then report the untouched hold-out exactly once.
val_scores = model.predict(val_ds, verbose=0).ravel()
def metric_dict(y, scores, threshold):
    pred = scores >= threshold
    tp, tn = int(((y == 1) & pred).sum()), int(((y == 0) & ~pred).sum())
    fp, fn = int(((y == 0) & pred).sum()), int(((y == 1) & ~pred).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {'samples': int(len(y)), 'truePositive': tp, 'trueNegative': tn, 'falsePositive': fp, 'falseNegative': fn, 'accuracy': round((tp + tn) / len(y), 4), 'precision': round(precision, 4), 'recall': round(recall, 4), 'falsePositiveRate': round(fp / (fp + tn), 4) if fp + tn else 0.0}
candidates = [(t, metric_dict(val_y, val_scores, t)) for t in np.arange(0.05, 0.96, 0.01)]
eligible = [(t, m) for t, m in candidates if m['falsePositiveRate'] <= MAX_FALSE_POSITIVE_RATE]
assert eligible, 'No validation threshold meets the FPR gate; collect better data rather than exporting.'
THRESHOLD, _ = max(eligible, key=lambda item: 2 * item[1]['precision'] * item[1]['recall'] / max(item[1]['precision'] + item[1]['recall'], 1e-9))
holdout_scores = model.predict(test_ds, verbose=0).ravel()
HOLDOUT = metric_dict(test_y, holdout_scores, THRESHOLD)
print('threshold:', THRESHOLD, 'hold-out:', HOLDOUT)
assert HOLDOUT['accuracy'] >= MIN_VALIDATION_ACCURACY and HOLDOUT['falsePositiveRate'] <= MAX_FALSE_POSITIVE_RATE, 'Release gate failed. Do not deploy this model.'


In [ ]:
# Export the exact NETRA-KERAS-1 artefacts. Copy both files into backend/data/netra/models/.
from datetime import datetime, timezone
OUT = ROOT / 'netra_export'
OUT.mkdir(exist_ok=True)
artifact = OUT / 'active_model.keras'
model.save(artifact)
artifact_sha = hashlib.sha256(artifact.read_bytes()).hexdigest()
provenance = ({'name': DATASET_NAME, 'url': DATASET_URL, 'license': DATASET_LICENSE, 'use': 'research-only; not RBI-certified'} if DATASET_MODE == 'kaggle_research' else {'name': DATASET_NAME, 'authorisedSourceCase': AUTHORISED_SOURCE_CASE, 'use': 'authorised collection; validate legal and retention basis before deployment'})
card = {'modelName': 'netra-ficn-cnn-v1', 'version': 'NETRA-KERAS-1', 'format': 'NETRA-KERAS-1', 'artifactFile': 'active_model.keras', 'modelSha256': artifact_sha, 'trainedAt': datetime.now(timezone.utc).isoformat(), 'datasetName': DATASET_NAME, 'datasetManifestSha256': MANIFEST_SHA256, 'datasetProvenance': provenance, 'trainingSamples': int(len(train_p)), 'holdoutMetrics': HOLDOUT, 'counterfeitThreshold': float(THRESHOLD), 'inputSize': list(IMAGE_SIZE), 'denominations': sorted(set(r['denomination'] for r in records)), 'limitations': ['Image-only classifier; it cannot verify UV, tactile, paper substrate, watermark-through-light or forensic features.', 'Research data is not RBI certification. Human review is mandatory before action.']}
(OUT / 'active_model_card.json').write_text(json.dumps(card, indent=2, sort_keys=True), encoding='utf-8')
shutil.make_archive('/content/netra_ficn_export', 'zip', OUT)
print(json.dumps(card, indent=2))
from google.colab import files
files.download('/content/netra_ficn_export.zip')


## Deploy

1. Extract `netra_ficn_export.zip`.
2. Copy `active_model.keras` and `active_model_card.json` to `backend/data/netra/models/` (or your configured `NETRA_MODEL_DIR`).
3. Install backend requirements, including `tensorflow-cpu`, and restart the API.
4. Confirm `GET /api/v1/netra/model/status` reports `ready: true`; then run `POST /api/v1/netra/model/evaluate` against a separately authorised test manifest.

Do not use the research Kaggle candidate for banking, law-enforcement action, or commercial use without confirming its licence and replacing it with authorised, representative genuine/FICN data.